<!-- NOTEBOOK_METADATA source: "⚠️ Jupyter Notebook" title: "Measure and optimize document extraction with Langfuse Experiments" sidebarTitle: "Structured Output Extraction" description: "Extract nested insurance claims with OpenAI structured outputs, then iterate with the Langfuse Experiments SDK using layered metrics and small-n significance tests." category: "Evaluation" -->

# Measure and optimize document extraction with Langfuse Experiments

Structured extraction returns a nested, typed object: strings, dates, numbers, enums, and lists of sub-objects. Correctness is not one number. A single accuracy score hides which fields failed, and "the output looks right" is not something you can optimize.

This guide extracts insurance claims with [OpenAI structured outputs](https://platform.openai.com/docs/guides/structured-outputs) and iterates with the [Langfuse Experiments SDK](https://langfuse.com/docs/evaluation/experiments/experiments-via-sdk). Each round changes one lever (schema, prompt, few-shot, architecture, or model), scores the same 30 claims, and checks whether the move is real before you keep it.

You will also see a change that introduces a regression, and a variant that looks better until you replicate it.

**Stack:** OpenAI structured outputs, Langfuse Tracing, Experiments, and Prompt Management. The scorer reports valid-JSON %, field accuracy by type, object F1, latency, and USD/doc. At n=30 we use a paired McNemar test and Wilson confidence intervals.

> **Credits.** Dataset, schema, and baseline comparison are adapted from Cleanlab's structured-output [benchmark](https://cleanlab.ai/blog/structured-output-benchmark/) ([repo](https://github.com/cleanlab/structured-output-benchmark/tree/main/insurance_claims), [dataset](https://huggingface.co/datasets/Cleanlab/insurance-claims-extraction)). Their `schema.py` is our `descriptive` schema. We extend their `utils.py` comparison with type-aware matching, object F1, and cost/latency.


## 1. Setup

Install the dependencies and set environment variables. Create API keys in [Langfuse Cloud](https://cloud.langfuse.com) or your [self-hosted](https://langfuse.com/self-hosting) project.


In [ ]:
%pip install -q "langfuse>=4.14" "openai>=3" pydantic pandas

In [ ]:
import os

# Get keys for your project from the project settings page: https://cloud.langfuse.com
os.environ.setdefault("LANGFUSE_PUBLIC_KEY", "pk-lf-...");
os.environ.setdefault("LANGFUSE_SECRET_KEY", "sk-lf-...");
os.environ.setdefault("LANGFUSE_BASE_URL", "https://cloud.langfuse.com"); # 🇪🇺 EU region
# 🇺🇸 US region: https://us.cloud.langfuse.com

# OpenAI API key
os.environ.setdefault("OPENAI_API_KEY", "sk-proj-...");

In [ ]:
from langfuse import get_client

langfuse = get_client()
assert langfuse.auth_check(), "Langfuse auth failed — check your keys / host"
print("Langfuse auth OK")

## 2. Schema as the extraction contract

The schema is half the spec. We keep the same nested structure in two variants:

- **bare**: types and constraints only. No field descriptions.
- **descriptive**: the same structure, plus `description=` text for ID formats and standardized values.

That lets us test one question later: do schema descriptions change what the model emits? Each claim has `header`, optional `policy_details`, optional `insured_objects[]` (a list of sub-objects, the hard part), and `incident_description`.


In [ ]:
from __future__ import annotations
from datetime import date
from typing import List, Literal, Optional
from pydantic import BaseModel, ConfigDict, Field

# ---- shared enum aliases (defined once -> both variants stay identical) -------
Channel = Literal["Email", "Phone", "Portal", "In-Person"]
CoverageType = Literal["Property", "Auto", "Liability", "Health", "Travel", "Other"]
ObjectType = Literal["Vehicle", "Building", "Person", "Other"]
IncidentType = Literal[
    "rear_end_collision", "side_impact_collision", "head_on_collision", "parking_lot_collision",
    "house_fire", "kitchen_fire", "electrical_fire", "burst_pipe_flood", "storm_damage", "roof_leak",
    "slip_and_fall", "property_injury", "product_liability", "theft_burglary", "vandalism",
]
LocationType = Literal[
    "intersection", "highway", "parking_lot", "driveway", "residential_street",
    "residence_interior", "residence_exterior", "commercial_property", "public_property",
]
_STRICT = ConfigDict(extra="forbid")

In [ ]:
# ---- BARE variant: structure + constraints only ------------------------------
class ClaimHeaderBare(BaseModel):
    model_config = _STRICT
    claim_id: str
    report_date: date
    incident_date: date
    reported_by: str = Field(..., min_length=1)
    channel: Channel

class PolicyDetailsBare(BaseModel):
    model_config = _STRICT
    policy_number: str
    policyholder_name: str = Field(..., min_length=1)
    coverage_type: CoverageType
    effective_date: date
    expiration_date: date

class InsuredObjectBare(BaseModel):
    model_config = _STRICT
    object_id: str
    object_type: ObjectType
    make_model: Optional[str] = None
    year: Optional[int] = None
    location_address: Optional[str] = None
    estimated_value: Optional[int] = None

class IncidentDescriptionBare(BaseModel):
    model_config = _STRICT
    incident_type: IncidentType
    location_type: LocationType
    estimated_damage_amount: Optional[int] = None
    police_report_number: Optional[str] = None

class InsuranceClaimBare(BaseModel):
    model_config = _STRICT
    header: ClaimHeaderBare
    policy_details: Optional[PolicyDetailsBare] = None
    insured_objects: Optional[List[InsuredObjectBare]] = None
    incident_description: IncidentDescriptionBare

In [ ]:
# ---- DESCRIPTIVE variant: same structure + rich descriptions (the "spec") ----
class ClaimHeader(BaseModel):
    model_config = _STRICT
    claim_id: str = Field(..., description="Claim ID in format CLM-XXXXXX, where X is a digit")
    report_date: date = Field(..., description="Date claim was reported (ISO 8601)")
    incident_date: date = Field(..., description="Date incident occurred (ISO 8601)")
    reported_by: str = Field(..., min_length=1, description="Full name of person reporting the claim")
    channel: Channel = Field(..., description="Channel used to report the claim")

class PolicyDetails(BaseModel):
    model_config = _STRICT
    policy_number: str = Field(..., description="Policy number in format POL-XXXXXXXXX, where X is a digit")
    policyholder_name: str = Field(..., min_length=1, description="Full legal name on the policy")
    coverage_type: CoverageType = Field(..., description="Type of insurance coverage")
    effective_date: date = Field(..., description="Policy effective start date (ISO 8601)")
    expiration_date: date = Field(..., description="Policy expiration end date (ISO 8601)")

class InsuredObject(BaseModel):
    model_config = _STRICT
    object_id: str = Field(..., description=(
        "Unique identifier for the insured object. For vehicles use VIN format "
        "(e.g., VIN12345678901234567). For buildings use PROP-XXXXXX. For liability use "
        "LIAB-XXXXXX. Otherwise use OBJ-XXXXXX, where X is a digit"))
    object_type: ObjectType = Field(..., description="Type of insured object")
    make_model: Optional[str] = Field(None, description=(
        "Make and model for vehicles (use standardized manufacturer names and models), "
        "or building type for property"))
    year: Optional[int] = Field(None, description="Year for vehicles or year built for buildings")
    location_address: Optional[str] = Field(None, description="Full street address where object is located")
    estimated_value: Optional[int] = Field(None, description="Estimated value in USD, no currency symbol")

class IncidentDescription(BaseModel):
    model_config = _STRICT
    incident_type: IncidentType = Field(..., description="Specific standardized incident type")
    location_type: LocationType = Field(..., description="Standardized location type where it occurred")
    estimated_damage_amount: Optional[int] = Field(None, description="Estimated damage in USD, no symbol")
    police_report_number: Optional[str] = Field(None, description="Police report number if applicable")

class InsuranceClaim(BaseModel):
    model_config = _STRICT
    header: ClaimHeader = Field(..., description="Basic claim information")
    policy_details: Optional[PolicyDetails] = Field(None, description="Policy information if available")
    insured_objects: Optional[List[InsuredObject]] = Field(None, description="Insured objects, if applicable")
    incident_description: IncidentDescription = Field(..., description="Structured incident details")

SCHEMAS = {"bare": InsuranceClaimBare, "descriptive": InsuranceClaim}

In [ ]:
# Peek at the strict JSON schema OpenAI will enforce.
from openai.lib._pydantic import to_strict_json_schema
strict = to_strict_json_schema(InsuranceClaim)
print("sections:", list(strict["properties"].keys()))
print("incident_type enum:", len(IncidentType.__args__), "values | location_type:", len(LocationType.__args__))

## 3. Load the data and upload a Langfuse dataset

The dataset is 30 synthetic claims from the [Cleanlab structured-output benchmark](https://cleanlab.ai/blog/structured-output-benchmark/): free-text narratives plus nested ground-truth dicts. The `claim_text` is LLM-generated, so treat this as a demo set, not production data. We attach metadata (`has_policy`, `n_objects`) so you can filter later.


In [ ]:
import ast
import pandas as pd

DATA_URL = "https://huggingface.co/datasets/Cleanlab/insurance-claims-extraction/resolve/main/insurance_claims_extraction.csv"
df = pd.read_csv(DATA_URL)
print("rows:", len(df), "| columns:", list(df.columns))
example_gt = ast.literal_eval(df["ground_truth"].iloc[0])   # ground_truth is a stringified dict
print("ground-truth sections:", list(example_gt.keys()))

<details>
<summary>Dataset slice helpers</summary>


In [ ]:
def count_leaves(d):
    # Non-null leaf values = the real signal density (extraction target size).
    if isinstance(d, dict):
        return sum(count_leaves(v) for v in d.values())
    if isinstance(d, list):
        return sum(count_leaves(v) for v in d)
    return 0 if d is None else 1

def derive_slices(gt):
    objects = gt.get("insured_objects") or []
    policy = gt.get("policy_details")
    return {"has_policy": policy is not None, "has_objects": len(objects) > 0,
            "n_objects": len(objects), "coverage_type": (policy or {}).get("coverage_type"),
            "incident_type": gt.get("incident_description", {}).get("incident_type"),
            "leaf_field_count": count_leaves(gt),
            "provenance": "cleanlab/structured-output-benchmark (synthetic)"}

items = []
for i, row in df.iterrows():
    gt = ast.literal_eval(row["ground_truth"])
    items.append({"id": f"claim-{i:03d}", "input": row["claim_text"],
                  "expected_output": gt, "metadata": derive_slices(gt)})

import collections
print("has_policy:", sum(it["metadata"]["has_policy"] for it in items), "/", len(items))
print("n_objects:", dict(sorted(collections.Counter(it["metadata"]["n_objects"] for it in items).items())))


</details>


In [ ]:
DATASET_NAME = "insurance-claims"
langfuse.create_dataset(
    name=DATASET_NAME,
    description="Nested insurance-claim extraction (Cleanlab). 30 synthetic claims; verify before production.",
    metadata={"source": "Cleanlab/insurance-claims-extraction", "n_items": len(items)})
for it in items:
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME, id=f"{DATASET_NAME}:{it['id']}",
        input=it["input"], expected_output=it["expected_output"], metadata=it["metadata"])
langfuse.flush()
dataset = langfuse.get_dataset(DATASET_NAME)
dataset_items = sorted(dataset.items, key=lambda x: x.id)
print("uploaded & fetched:", len(dataset_items), "items")

![The insurance-claims dataset in Langfuse with inputs, expected outputs, and metadata slices](/images/cookbook/example-structured-output-extraction/dataset-items.png)


## 4. Layered extraction metrics

Do not collapse a nested, typed object into one accuracy number. The scorer uses this model:

1. **Valid JSON is a gate, not a score.** Structured outputs usually hit 100%. Parsing is not correctness.
2. **Pooled `field_acc` hides the work.** Easy copied fields drown out reasoned ones. Break scores out by section and field type.
3. **Compare by type.** Parse dates, tolerate numbers, normalize enums and names. Do not use `==` on everything.
4. **Lists of objects need precision, recall, and F1.** Match predicted objects to ground truth, then score fields inside matches. If the truth has zero objects and the model emits one, F1 is 0.
5. **Presence and content are different failures.** Missing `policy_details` is not the same as extracting the wrong policy number.
6. **Null rules.** A required field that is `None` in the truth is a data gap: skip it. An optional field that is `None` is gradable: the model must emit `null`. Inventing a value is wrong.
7. **Cost and latency are first-class.** Use p50/p95 for latency. Read USD/doc from Langfuse `trace.total_cost` (cache-aware) instead of a static price table.
8. **At n=20–50, prove the delta.** A 3-point move can be one document. Use a per-item binary, a paired McNemar test, a Wilson CI, and a replication before you ship.
9. **Aggregates find the smoke. Traces find the fire.** Open failing items in Langfuse. Every fix in this guide came from a trace.

> The scorer adapts Cleanlab's [`utils.py`](https://github.com/cleanlab/structured-output-benchmark/blob/main/insurance_claims/utils.py) comparison (per-section matching, normalized strings, object pairing) and adds typed date/number matching, object F1, ID-format checks, presence, latency, and cost.


<details>
<summary>Scorer implementation</summary>


In [ ]:
import re, math, statistics
from datetime import datetime
from itertools import combinations, permutations

OPTIONAL_LEAVES = {"header": set(), "policy_details": set(),
    "insured_objects": {"make_model", "year", "location_address", "estimated_value"},
    "incident_description": {"estimated_damage_amount", "police_report_number"}}
DATE_FIELDS = {"header.report_date", "header.incident_date",
               "policy_details.effective_date", "policy_details.expiration_date"}
NUMERIC_FIELDS = {"incident_description.estimated_damage_amount",
                  "insured_objects.year", "insured_objects.estimated_value"}
ENUM_FIELDS = {"header.channel", "policy_details.coverage_type", "insured_objects.object_type",
               "incident_description.incident_type", "incident_description.location_type"}
ID_REGEX = {"header.claim_id": r"^CLM-\d{6}$", "policy_details.policy_number": r"^POL-\d{9}$"}
OBJECT_ID_REGEX = {"Vehicle": r"^VIN[A-Z0-9]{17}$", "Building": r"^PROP-\d{6}$",
                   "Person": r"^(LIAB|OBJ)-\d{6}$", "Other": r"^(LIAB|OBJ)-\d{6}$"}
NUMERIC_ABS_TOL, NUMERIC_REL_TOL, OBJECT_MATCH_THRESHOLD = 1.0, 0.01, 0.5

def norm_str(v):
    s = re.sub(r"[^\w\s]", "", str(v).strip().lower())
    return re.sub(r"\s+", "", s)

def to_date(v):
    if isinstance(v, date):
        return v
    s = str(v).strip()
    try:
        return date.fromisoformat(s[:10])
    except ValueError:
        for fmt in ("%m/%d/%Y", "%d/%m/%Y", "%B %d, %Y", "%b %d, %Y", "%Y/%m/%d"):
            try:
                return datetime.strptime(s, fmt).date()
            except ValueError:
                continue
    return None

def match_date(gt, pred):
    a, b = to_date(gt), to_date(pred)
    return a == b if (a and b) else norm_str(gt) == norm_str(pred)

def match_number(gt, pred):
    try:
        g, p = float(gt), float(pred)
    except (TypeError, ValueError):
        return norm_str(gt) == norm_str(pred)
    return abs(g - p) <= max(NUMERIC_ABS_TOL, NUMERIC_REL_TOL * abs(g))

def match_value(path, gt, pred):
    if path in DATE_FIELDS:
        return match_date(gt, pred)
    if path in NUMERIC_FIELDS:
        return match_number(gt, pred)
    return norm_str(gt) == norm_str(pred)   # enums + free strings

def is_optional(section, field):
    return field in OPTIONAL_LEAVES.get(section, set())

def compare_section(section, gt, pred):
    pred = pred or {}
    correct = total = enum_c = enum_t = date_c = date_t = num_c = num_t = 0
    for field, gt_val in gt.items():
        path = f"{section}.{field}"
        if gt_val is None:
            if not is_optional(section, field):
                continue                       # required + null => not gradable (data gap)
            total += 1; correct += int(pred.get(field) is None)
            continue
        ok = (pred.get(field) is not None) and match_value(path, gt_val, pred.get(field))
        total += 1; correct += int(ok)
        if path in ENUM_FIELDS:   enum_t += 1; enum_c += int(ok)
        elif path in DATE_FIELDS: date_t += 1; date_c += int(ok)
        elif path in NUMERIC_FIELDS: num_t += 1; num_c += int(ok)
    return dict(correct=correct, total=total, enum_c=enum_c, enum_t=enum_t,
                date_c=date_c, date_t=date_t, num_c=num_c, num_t=num_t)

def object_similarity(gt_obj, pred_obj):
    r = compare_section("insured_objects", gt_obj, pred_obj)
    return r["correct"] / r["total"] if r["total"] else 0.0

def optimal_pairing(gt_objs, pred_objs):
    # pairing[i] = index in pred matched to gt[i] (or None), maximizing similarity.
    n_gt, n_pred = len(gt_objs), len(pred_objs)
    if not n_gt or not n_pred:
        return [None] * n_gt
    if max(n_gt, n_pred) > 6:                  # greedy fallback (never hit on this data)
        used, pairing = set(), []
        for g in gt_objs:
            best, bi = -1.0, None
            for j, p in enumerate(pred_objs):
                if j in used:
                    continue
                s = object_similarity(g, p)
                if s > best:
                    best, bi = s, j
            if bi is not None:
                used.add(bi)
            pairing.append(bi)
        return pairing
    sim = [[object_similarity(g, p) for p in pred_objs] for g in gt_objs]
    best_score, best = -1.0, [None] * n_gt
    for k in range(min(n_gt, n_pred) + 1):
        for gt_idx in combinations(range(n_gt), k):
            for pred_idx in permutations(range(n_pred), k):
                pairing, score = [None] * n_gt, 0.0
                for a, gi in enumerate(gt_idx):
                    pairing[gi] = pred_idx[a]; score += sim[gi][pred_idx[a]]
                if score > best_score:
                    best_score, best = score, pairing
    return best

def score_objects(gt_objs, pred_objs):
    gt_objs, pred_objs = gt_objs or [], pred_objs or []
    n_gt, n_pred = len(gt_objs), len(pred_objs)
    if n_gt == 0 and n_pred == 0:
        return dict(precision=1.0, recall=1.0, f1=1.0, tp=0, field_correct=0, field_total=0, count_correct=True)
    pairing = optimal_pairing(gt_objs, pred_objs)
    tp = field_correct = field_total = 0
    for gi, pj in enumerate(pairing):
        if pj is None:
            continue
        r = compare_section("insured_objects", gt_objs[gi], pred_objs[pj])
        if (r["correct"] / r["total"] if r["total"] else 0.0) >= OBJECT_MATCH_THRESHOLD:
            tp += 1; field_correct += r["correct"]; field_total += r["total"]
    precision = tp / n_pred if n_pred else (1.0 if n_gt == 0 else 0.0)
    recall = tp / n_gt if n_gt else (1.0 if n_pred == 0 else 0.0)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return dict(precision=precision, recall=recall, f1=f1, tp=tp,
                field_correct=field_correct, field_total=field_total, count_correct=(n_gt == n_pred))

def score_id_formats(pred):
    ok = total = 0
    cid = pred.get("header", {}).get("claim_id")
    if cid is not None:
        total += 1; ok += int(bool(re.match(ID_REGEX["header.claim_id"], str(cid))))
    pol = pred.get("policy_details")
    if pol and pol.get("policy_number") is not None:
        total += 1; ok += int(bool(re.match(ID_REGEX["policy_details.policy_number"], str(pol["policy_number"]))))
    for obj in (pred.get("insured_objects") or []):
        oid = obj.get("object_id")
        if oid is None:
            continue
        rgx = OBJECT_ID_REGEX.get(obj.get("object_type"), r"^(OBJ|LIAB)-\d{6}$")
        total += 1; ok += int(bool(re.match(rgx, str(oid))))
    return dict(ok=ok, total=total)

def compare_claim(pred, gt):
    if pred is None:
        return dict(valid_json=False)
    header = compare_section("header", gt.get("header", {}), pred.get("header"))
    incident = compare_section("incident_description", gt.get("incident_description", {}), pred.get("incident_description"))
    gt_policy, pred_policy = gt.get("policy_details"), pred.get("policy_details")
    policy = compare_section("policy_details", gt_policy, pred_policy) if gt_policy \
        else dict(correct=0, total=0, enum_c=0, enum_t=0, date_c=0, date_t=0, num_c=0, num_t=0)
    objs = score_objects(gt.get("insured_objects"), pred.get("insured_objects"))
    f_correct = header["correct"] + incident["correct"] + policy["correct"] + objs["field_correct"]
    f_total = header["total"] + incident["total"] + policy["total"] + objs["field_total"]
    for gobj in (gt.get("insured_objects") or [])[objs["tp"]:]:
        f_total += compare_section("insured_objects", gobj, {})["total"]
    enum_c = header["enum_c"] + incident["enum_c"] + policy["enum_c"]; enum_t = header["enum_t"] + incident["enum_t"] + policy["enum_t"]
    date_c = header["date_c"] + incident["date_c"] + policy["date_c"]; date_t = header["date_t"] + incident["date_t"] + policy["date_t"]
    num_c = header["num_c"] + incident["num_c"] + policy["num_c"]; num_t = header["num_t"] + incident["num_t"] + policy["num_t"]
    pairing = optimal_pairing(gt.get("insured_objects") or [], pred.get("insured_objects") or [])
    for gi, pj in enumerate(pairing):
        if pj is None:
            continue
        r = compare_section("insured_objects", gt["insured_objects"][gi], (pred.get("insured_objects") or [])[pj])
        enum_c += r["enum_c"]; enum_t += r["enum_t"]; num_c += r["num_c"]; num_t += r["num_t"]
    ids = score_id_formats(pred)
    presence = [bool(gt_policy) == bool(pred_policy), bool(gt.get("insured_objects")) == bool(pred.get("insured_objects"))]
    ratio = lambda c, t: (c / t) if t else None
    return dict(valid_json=True,
        field_accuracy_overall=ratio(f_correct, f_total), header_accuracy=ratio(header["correct"], header["total"]),
        policy_accuracy=ratio(policy["correct"], policy["total"]), incident_accuracy=ratio(incident["correct"], incident["total"]),
        enum_accuracy=ratio(enum_c, enum_t), date_accuracy=ratio(date_c, date_t),
        numeric_accuracy=ratio(num_c, num_t), id_format_ok=ratio(ids["ok"], ids["total"]),
        objects_count_correct=objs["count_correct"], objects_precision=objs["precision"], objects_recall=objs["recall"],
        objects_f1=objs["f1"], object_field_accuracy=ratio(objs["field_correct"], objs["field_total"]),
        section_presence_correct=sum(presence) / len(presence))


</details>


In [ ]:
from langfuse import Evaluation

_NUMERIC_METRICS = ["field_accuracy_overall", "header_accuracy", "policy_accuracy", "incident_accuracy",
                    "enum_accuracy", "date_accuracy", "numeric_accuracy", "id_format_ok",
                    "objects_precision", "objects_recall", "objects_f1", "object_field_accuracy",
                    "section_presence_correct"]

def extraction_evaluator(*, input, output, expected_output, metadata=None, **kwargs):
    if not isinstance(output, dict):
        return [Evaluation(name="valid_json", value=False, data_type="BOOLEAN")]
    b = compare_claim(output.get("prediction"), expected_output)
    evals = [Evaluation(name="valid_json", value=bool(b.get("valid_json")), data_type="BOOLEAN")]
    if not b.get("valid_json"):
        evals.append(Evaluation(name="field_accuracy_overall", value=0.0, data_type="NUMERIC"))
    else:
        for m in _NUMERIC_METRICS:
            if b.get(m) is not None:
                evals.append(Evaluation(name=m, value=float(b[m]), data_type="NUMERIC"))
        evals.append(Evaluation(name="objects_count_correct", value=bool(b["objects_count_correct"]), data_type="BOOLEAN"))
    if output.get("latency_ms") is not None:   # cost is NOT scored — Langfuse tracks it natively
        evals.append(Evaluation(name="latency_ms", value=float(output["latency_ms"]), data_type="NUMERIC"))
    return evals

def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0, 0.0)
    p = k / n; denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    half = (z * math.sqrt(p*(1-p)/n + z*z/(4*n*n))) / denom
    return (p, max(0.0, center - half), min(1.0, center + half))

def run_aggregates(*, item_results, **kwargs):
    from collections import defaultdict
    vals = defaultdict(list)
    for ir in item_results:
        for ev in (getattr(ir, "evaluations", None) or []):
            name = ev.get("name") if isinstance(ev, dict) else getattr(ev, "name", None)
            value = ev.get("value") if isinstance(ev, dict) else getattr(ev, "value", None)
            if name is not None and isinstance(value, (int, float, bool)):
                vals[name].append(float(value))
    out = []
    for name, xs in vals.items():
        if not xs or name == "latency_ms":          # latency -> p50, not mean
            continue
        out.append(Evaluation(name=f"mean_{name}", value=sum(xs)/len(xs), data_type="NUMERIC"))
    if vals.get("latency_ms"):
        out.append(Evaluation(name="p50_latency_ms", value=statistics.median(vals["latency_ms"]), data_type="NUMERIC"))
    if "valid_json" in vals:
        p, lo, hi = wilson_ci(int(sum(vals["valid_json"])), len(vals["valid_json"]))
        out.append(Evaluation(name="valid_json_ci95", value=p, data_type="NUMERIC",
                              comment=f"Wilson 95% CI [{lo:.3f}, {hi:.3f}] (n={len(vals['valid_json'])})"))
    return out

## 5. Experiment harness

`run_round(...)` runs one variant with `dataset.run_experiment`, attaches the evaluators, and adds a row to a comparison table.

- **Cost** comes from each trace's `total_cost` in Langfuse, not a price table. `native_cost_mean` retries briefly because cost is computed at ingestion.
- **Prompts** live in Langfuse as `extraction-system`. Each run links to the prompt version it used. We seed v1 (terse) first.


In [ ]:
from functools import lru_cache
from langfuse.openai import openai
import time

PROMPT_NAME = "extraction-system"
TERSE_V1 = "Extract the insurance claim from the text into the structured schema."

@lru_cache(maxsize=16)
def get_system_prompt(version):
    return langfuse.get_prompt(PROMPT_NAME, version=version)

def seed_prompt(version, text, labels, commit_message):
    # Idempotent: prompt versions are immutable, so only create what doesn't exist yet.
    metas = langfuse.api.prompts.list(name=PROMPT_NAME).data
    if any(version in (m.versions or []) for m in metas):
        print(f"prompt v{version} already exists")
        return
    langfuse.create_prompt(name=PROMPT_NAME, prompt=text, type="text",
                           labels=labels, commit_message=commit_message)
    get_system_prompt.cache_clear()
    print(f"created prompt v{version}")

# Seed v1 (terse baseline). Later we'll add v2/v3 when the results tell us to.
seed_prompt(1, TERSE_V1, ["baseline"], "v1: terse baseline")

In [ ]:
# Message builder — used by every round (few-shot example is added later, when its round comes).
def build_messages(system_text, claim_text, few_shot):
    msgs = [{"role": "system", "content": system_text}]
    if few_shot:
        msgs += [{"role": "user", "content": FEW_SHOT_USER}, {"role": "assistant", "content": FEW_SHOT_ASSISTANT}]
    msgs.append({"role": "user", "content": claim_text})
    return msgs

In [ ]:
# The task: one extraction. Single-pass by default; the decomposition idea splits it
# into one call per section. Either way it returns the dict our evaluator reads.
def extract(cfg, claim_text, name):
    system_text = TERSE_V1 if cfg["prompt_version"] is None else get_system_prompt(cfg["prompt_version"]).compile()
    prompt_obj = None if cfg["prompt_version"] is None else get_system_prompt(cfg["prompt_version"])
    if not cfg["decompose"]:
        t0 = time.perf_counter()
        c = openai.chat.completions.parse(model=cfg["model"],
            messages=build_messages(system_text, claim_text, cfg["few_shot"]),
            response_format=SCHEMAS[cfg["schema"]], name=name, langfuse_prompt=prompt_obj, temperature=0)
        parsed = c.choices[0].message.parsed
        return {"prediction": parsed.model_dump(mode="json") if parsed else None,
                "valid_json": parsed is not None, "latency_ms": (time.perf_counter()-t0)*1000,
                "model": cfg["model"], "n_calls": 1}
    sections = [("header", ClaimHeader), ("incident_description", IncidentDescription),
                ("policy", _PolicyWrapper), ("objects", _ObjectsWrapper)]
    parts, valid, t0 = {}, True, time.perf_counter()
    for sec, cls in sections:
        c = openai.chat.completions.parse(model=cfg["model"],
            messages=build_messages(system_text, claim_text, False),
            response_format=cls, name=f"{name}:{sec}", langfuse_prompt=prompt_obj, temperature=0)
        p = c.choices[0].message.parsed
        valid = valid and p is not None
        parts[sec] = p.model_dump(mode="json") if p else None
    pred = None
    if valid:
        pred = {"header": parts["header"], "policy_details": (parts["policy"] or {}).get("policy_details"),
                "insured_objects": (parts["objects"] or {}).get("insured_objects"),
                "incident_description": parts["incident_description"]}
    return {"prediction": pred, "valid_json": valid, "latency_ms": (time.perf_counter()-t0)*1000,
            "model": cfg["model"], "n_calls": len(sections)}

<details>
<summary>Aggregation helpers</summary>


In [ ]:
# Aggregation helpers for the growing comparison table (native cost from Langfuse).
def ev_dict(evs):
    return {getattr(e, "name", None): getattr(e, "value", None) for e in (evs or [])}

def native_cost_mean(item_results, wait=3.0, rounds=2):
    """Mean per-doc USD from Langfuse's native trace.total_cost (cache-aware).
    Cost is computed at ingestion, so we re-poll only the traces still missing it.
    Worst case ~6s; never blocks the metrics table."""
    langfuse.flush()
    pending = {ir.trace_id: None for ir in item_results if getattr(ir, "trace_id", None)}
    for r in range(rounds + 1):
        for tid in [t for t, c in pending.items() if c is None]:
            try:
                pending[tid] = langfuse.api.trace.get(tid).total_cost
            except Exception:
                pass
        if all(c is not None for c in pending.values()):
            break
        if r < rounds:
            time.sleep(wait)
    costs = [c for c in pending.values() if c is not None]
    return sum(costs) / len(costs) if costs else float("nan")


</details>


In [ ]:
HEADLINE = [("valid_json", "valid_json%"), ("field_accuracy_overall", "field_acc"),
            ("objects_f1", "objects_F1"), ("id_format_ok", "id_format%"),
            ("date_accuracy", "date_acc"), ("enum_accuracy", "enum_acc")]

results, summaries = {}, []   # accumulate across rounds

def run_round(name, *, model="gpt-4o-mini", schema="descriptive", prompt_version=None,
              few_shot=False, decompose=False, sample=None):
    cfg = dict(model=model, schema=schema, prompt_version=prompt_version, few_shot=few_shot, decompose=decompose)
    def task(*, item, **kwargs):
        text = item.input if hasattr(item, "input") else item["input"]
        return extract(cfg, text, name)
    data = None
    if sample:
        sub = dataset_items[:sample]
        data = [{"input": it.input, "expected_output": it.expected_output, "metadata": it.metadata} for it in sub]
    kw = dict(name=name, task=task, evaluators=[extraction_evaluator],
              run_evaluators=[run_aggregates], max_concurrency=8)
    res = langfuse.run_experiment(data=data, **kw) if data else dataset.run_experiment(**kw)
    langfuse.flush()
    results[name] = res
    rows = [ev_dict(ir.evaluations) for ir in res.item_results]
    outs = [ir.output if isinstance(ir.output, dict) else {} for ir in res.item_results]
    mean = lambda m: (sum(float(r[m]) for r in rows if isinstance(r.get(m), (int, float, bool)))
                      / max(1, sum(isinstance(r.get(m), (int, float, bool)) for r in rows)))
    lat = [o["latency_ms"] for o in outs if o.get("latency_ms") is not None]
    s = {"variant": name}
    for m, lbl in HEADLINE:
        s[lbl] = round(mean(m), 3)
    s["p50_lat_ms"] = round(statistics.median(lat)) if lat else None
    s["usd/1k"] = round(native_cost_mean(res.item_results) * 1000, 3)
    s["_id_perfect"] = [1 if r.get("id_format_ok") == 1.0 else 0 for r in rows]
    s["_keys"] = [getattr(ir.item, "id", None) or getattr(ir.item, "input", None) for ir in res.item_results]
    summaries.append(s)
    return comparison_table()

def comparison_table():
    cols = ["variant"] + [lbl for _, lbl in HEADLINE] + ["p50_lat_ms", "usd/1k"]
    return pd.DataFrame([{k: s[k] for k in cols} for s in summaries])

SAMPLE = None   # set to a small int to dry-run on fewer items while developing

## 6. Round 1: baseline

Bare schema, terse prompt v1, `gpt-4o-mini`. This is the floor.


In [ ]:
run_round("V1_baseline", model="gpt-4o-mini", schema="bare", prompt_version=1, sample=SAMPLE)

Read `field_acc` (~0.9) as a pool of every leaf, not as "almost done."

The main miss is `id_format%`, almost entirely `object_id`. Open the traces: the model numbers objects `"1"`, `"2"`, and so on. The bare schema gives a type but no format, and the text rarely has an id, so the model invents the obvious default. It also invents objects on claims that have none (`objects_F1 = 0`) and mangles a few non-ISO dates.

The next cheaper lever is the schema descriptions.


## 7. Round 2: schema descriptions

Same terse prompt and `gpt-4o-mini`. Only the schema changes (bare to descriptive).


In [ ]:
run_round("V2_schema_spec", model="gpt-4o-mini", schema="descriptive", prompt_version=1, sample=SAMPLE)

![Compare runs view in Langfuse: V1 baseline vs V2 schema-spec side by side](/images/cookbook/example-structured-output-extraction/compare-runs-v1-v2.png)


### Is the difference real at n=30?

A few points can be one document. Before treating V2 as a win, we score a per-claim binary: did every ID in this claim have a valid format? Then we run a paired McNemar test (only claims where the variants disagree) plus a Wilson CI. We reuse this after later rounds.


<details>
<summary>Significance-test helpers</summary>


In [ ]:
def mcnemar_exact(a, b):
    bb = sum(1 for x, y in zip(a, b) if x == 0 and y == 1)
    cc = sum(1 for x, y in zip(a, b) if x == 1 and y == 0)
    n = bb + cc
    if n == 0:
        return {"b": bb, "c": cc, "p_value": 1.0}
    k = min(bb, cc)
    return {"b": bb, "c": cc, "p_value": min(1.0, 2*sum(math.comb(n, i) for i in range(k+1))*(0.5**n))}

def align(a_name, b_name):
    A = next(s for s in summaries if s["variant"] == a_name)
    B = next(s for s in summaries if s["variant"] == b_name)
    ad, bd = dict(zip(A["_keys"], A["_id_perfect"])), dict(zip(B["_keys"], B["_id_perfect"]))
    common = [k for k in ad if k in bd]
    return [ad[k] for k in common], [bd[k] for k in common]

def compare_id_format(a_name, b_name):
    a, b = align(a_name, b_name); mc = mcnemar_exact(a, b)
    _, lo, hi = wilson_ci(sum(b), len(b))
    print(f"{a_name} -> {b_name}: id-perfect {sum(a)/len(a):.2f} -> {sum(b)/len(b):.2f} "
          f"[Wilson {lo:.2f},{hi:.2f}]  discordant b={mc['b']} c={mc['c']}  p={mc['p_value']:.4f}"
          + ("  ** significant" if mc['p_value'] < 0.05 else f"  (not significant, n={len(b)})"))


</details>


In [ ]:
compare_id_format("V1_baseline", "V2_schema_spec")

**Finding.** Descriptions move `id_format%` a lot, and the wins are one-sided: about 10–12 claims go wrong to right and about 0 the other way. That is a real effect.

Do not take the full jump at face value. Part of the gain is well-formatted IDs the model was forced to invent. `enum_acc` is flat, the same claims still hallucinate objects, and the richer schema costs a little more. A description can teach a format. It cannot teach a judgment, or invent a value that is not in the text.


![Experiment item in Langfuse where the model was forced to invent an object_id](/images/cookbook/example-structured-output-extraction/forced-object-id-trace.png)


### What `id_format%` is actually scoring

The schema (taken from the Cleanlab benchmark) makes `object_id` a required string. Of the 27 ground-truth objects across 30 claims, 11 have `object_id = null`. The model cannot be correct on those: the schema forbids `null`, so it has to invent an ID.

What that does to the scores:

- **Accuracy and F1 stay honest.** The scorer skips a required-but-null field as not gradable. An invented `object_id` does not count against `field_acc`, `object_field_accuracy`, or `objects_F1`.
- **`id_format%` does not.** It regex-checks every emitted ID. Split the 27 objects by whether a real ID exists:

| object_id with valid format | real GT id (16) | null GT id, forced (11) |
|---|---|---|
| **V1 (bare schema)** | 11 / 16 | 0 / 11 |
| **V2 (descriptive schema)** | 15 / 16 | 6 / 11 |

Valid formats went 11 to 21. Of that gain of 10: +4 is real (true IDs getting a `VIN`/`PROP` prefix, for example claim-017), and +6 is the model going from `"1"` to a well-formatted `PROP-000001` for an ID that should not exist. Six of the ten object-id gains are formatting hallucinations. The V1 to V2 `id_format%` jump is real, but inflated.

The schema fix is to make `object_id` `Optional[str] = None` so the model can return `null`. The broader point: evals score the model, and they also surface bugs in the schema and the metric. Reading the table without the traces would have shipped the wrong conclusion.


## 8. Round 3: prompt rules

Add an explicit instruction set as prompt v2 (`rules`). Same descriptive schema. The only change versus V2 is the prompt.


In [ ]:
RULES_V2 = """\
Extract the insurance claim into the structured schema. Follow these rules:
- IDs: claim_id must match CLM-XXXXXX (6 digits); policy_number must match POL-XXXXXXXXX (9 digits).
  For insured objects use the format implied by object_type: VINxx... for vehicles, PROP-XXXXXX for
  buildings, LIAB-XXXXXX for liability, OBJ-XXXXXX otherwise.
- Dates: output ISO 8601 (YYYY-MM-DD).
- Enums: choose the single closest standardized value; never invent new enum values.
- Do NOT invent information. If policy details are not present in the text, omit policy_details.
  If no insured objects are described, omit insured_objects. Leave optional fields null when absent.
- Extract values verbatim where possible; do not normalize names beyond what the schema requires."""


seed_prompt(2, RULES_V2, ["production"], "v2: explicit rules")

In [ ]:
run_round("V3_prompt_rules", model="gpt-4o-mini", schema="descriptive", prompt_version=2, sample=SAMPLE)

In [ ]:
compare_id_format("V2_schema_spec", "V3_prompt_rules")

**Finding.** Rules barely beat V2 on IDs, and the ID test is not significant. Schema descriptions already did that work. The cumulative gain versus V1 is still large.

The real move is object judgment: `objects_F1` jumps and several phantom-object claims drop. "Do not invent objects" is a decision a description cannot encode.

Watch `policy_acc`. It can drop because "don't invent / extract verbatim" gets applied to a field that is present. The model writes a placeholder such as `policyholder_name = "/null"` for a name it extracted correctly in V2. That is a regression to fix, not a ranking detail.


![V2 trace in Langfuse: policyholder_name extracted correctly](/images/cookbook/example-structured-output-extraction/policyholder-name-v2.png)


![V3 trace in Langfuse: the same claim now returns a '/null' placeholder for policyholder_name](/images/cookbook/example-structured-output-extraction/policyholder-name-v3-regression.png)


## 9. Audit what V3 broke

Check three things the scores pointed at: `location_type` enum confusion, remaining `object_id` format misses, and the `policyholder_name` placeholders.


<details>
<summary>Drill-down audit</summary>


In [ ]:
from collections import Counter

def enum_confusion(run, field_path):
    section, field = field_path.split(".", 1)
    conf = Counter()
    for ir in run.item_results:
        gt = ir.item.expected_output if hasattr(ir.item, "expected_output") else ir.item.get("expected_output")
        pred = ir.output.get("prediction") if isinstance(ir.output, dict) else None
        g = (gt.get(section) or {}).get(field) if gt else None
        p = (pred.get(section) or {}).get(field) if pred else None
        if g is not None:
            conf[(g, p if p is not None else "<missing>")] += 1
    return conf

def drill_into(run, label):
    print(f"{label} location_type confusions (ground_truth -> predicted):")
    for (g, p), n in sorted(enum_confusion(run, "incident_description.location_type").items(), key=lambda x: -x[1]):
        if g != p:
            print(f"  {g} -> {p}: {n}")
    print(f"\n{label} object_id format misses:")
    for ir in run.item_results:
        pred = ir.output.get("prediction") if isinstance(ir.output, dict) else None
        for obj in ((pred or {}).get("insured_objects") or []):
            oid, ot = obj.get("object_id"), obj.get("object_type")
            rgx = OBJECT_ID_REGEX.get(ot, r"^(OBJ|LIAB)-\d{6}$")
            if oid and not re.match(rgx, str(oid)):
                print(f"  {ot}: {oid!r}")
    print(f"\n{label} policyholder_name regressions (placeholder / truncated):")
    for ir in run.item_results:
        pred = ir.output.get("prediction") if isinstance(ir.output, dict) else None
        name = ((pred or {}).get("policy_details") or {}).get("policyholder_name")
        if name and (str(name).strip("/").lower() in ("null", "na", "") or len(str(name).split()) < 2):
            print(f"  {getattr(ir.item, 'id', '?')}: policyholder_name={name!r}")


</details>


In [ ]:
drill_into(results["V3_prompt_rules"], "V3")

## 10. Prompt v3: fix the regression

Append two rules: copy `policyholder_name` exactly, and only emit an insured object that is actually described. Save as `extraction-system` v3 and rerun. Prompt versions are immutable. A change is a new version, and the run links to it.


In [ ]:
RULES_V3 = RULES_V2 + """
- policyholder_name: copy the policyholder's full name exactly as written. NEVER output 'null', 'NA',
  '/', or any placeholder for a field that IS present.
- Insured objects: only emit an object explicitly described as insured or damaged. If none are, return
  an empty list — do NOT invent one to fill an id. When an object exists, its object_id uses the real
  value/format for its type; never copy the literal 'xxxx' placeholder."""


seed_prompt(3, RULES_V3, ["candidate"], "v3: copy name verbatim; stop re-inventing objects")

In [ ]:
run_round("V3_fixed", model="gpt-4o-mini", schema="descriptive", prompt_version=3, sample=SAMPLE)

In [ ]:
compare_id_format("V3_prompt_rules", "V3_fixed")

![The extraction-system prompt with versions v1-v3 in Langfuse Prompt Management](/images/cookbook/example-structured-output-extraction/prompt-versions.png)


**Finding.** The placeholder names come back as full names and `policy_acc` goes up. That is the metric this change targeted.

Headline numbers barely move, and `id_format%` / `field_acc` can wobble down run to run. The ID test above is a guardrail, not the proof. Read `policy_acc`.


## 11. Round 4: few-shot, then replicate

Add one worked example to the v2 rules prompt. Then run the same example on v3. The second run is the replication check.


In [ ]:
FEW_SHOT_USER = """\
[Phone call logged 2024-01-15]
Caller Tom Reyes reports a rear-end collision on the highway on 2024-01-14. Policy POL-100200300
(Auto, holder Tom Reyes), effective 2023-06-01 through 2024-06-01. Vehicle involved:
VIN1HGCM82633A004352, a 2019 Honda Accord, value about $18,000. Estimated damage $4,200.
Police report PR-2024-0114-77. Claim id CLM-553311."""
FEW_SHOT_ASSISTANT = """\
{"header": {"claim_id": "CLM-553311", "report_date": "2024-01-15", "incident_date": "2024-01-14",
"reported_by": "Tom Reyes", "channel": "Phone"},
"policy_details": {"policy_number": "POL-100200300", "policyholder_name": "Tom Reyes",
"coverage_type": "Auto", "effective_date": "2023-06-01", "expiration_date": "2024-06-01"},
"insured_objects": [{"object_id": "VIN1HGCM82633A004352", "object_type": "Vehicle",
"make_model": "Honda Accord", "year": 2019, "location_address": null, "estimated_value": 18000}],
"incident_description": {"incident_type": "rear_end_collision", "location_type": "highway",
"estimated_damage_amount": 4200, "police_report_number": "PR-2024-0114-77"}}"""

In [ ]:
run_round("V4_few_shot",       model="gpt-4o-mini", schema="descriptive", prompt_version=2, few_shot=True, sample=SAMPLE)
run_round("V4b_fewshot_on_fix", model="gpt-4o-mini", schema="descriptive", prompt_version=3, few_shot=True, sample=SAMPLE)
comparison_table()

**Finding.** One few-shot run can look like a breakthrough if `objects_F1` jumps. Replicate on the other prompt. If the same claims flip, it is n=30 noise, not an effect of the example.

The reproducible effect to watch is a regression: `date_acc` can drop on both prompts, and each call costs more.


## 12. Round 5: one call per section

Extract each section with its own sub-schema and stitch the parts. Same v2 rules prompt. Four calls instead of one.


In [ ]:
class _PolicyWrapper(BaseModel):
    model_config = _STRICT
    policy_details: Optional[PolicyDetails] = Field(None, description="Policy info if present, else null")
class _ObjectsWrapper(BaseModel):
    model_config = _STRICT
    insured_objects: Optional[List[InsuredObject]] = Field(None, description="Objects if present, else null")

In [ ]:
run_round("V5_decomposition", model="gpt-4o-mini", schema="descriptive", prompt_version=2, decompose=True, sample=SAMPLE)

**Finding.** Quality drops on most axes, at about 2× cost and latency. The isolated object call loses cross-section context and over-emits objects. An experiment can reject a modular design in one run.


## 13. Round 6: a larger model

Same v3 prompt. Swap `gpt-4o-mini` for `gpt-4o`.


In [ ]:
run_round("V6_gpt4o", model="gpt-4o", schema="descriptive", prompt_version=3, sample=SAMPLE)
comparison_table()

**Finding.** `gpt-4o` is better and often faster, at roughly 17–20× the price. At n=30 the per-metric deltas versus the tuned mini sit in the noise (McNemar not significant). It does separate on the hardest object-boundary claims.

The cheap tuned setup matches `gpt-4o` on almost everything. Pay for the larger model only if those edge cases matter.


## 14. Read the runs back (optional)

The Experiments SDK stores every score in Langfuse. This cell lists each run id from the API, which is useful for dashboards or CI.


In [ ]:
for s in summaries:
    run = langfuse.api.datasets.get_run(dataset_name=DATASET_NAME, run_name=results[s["variant"]].run_name)
    # run-level scores live on the dataset run; per-item scores live on each trace
    print(s["variant"], "→ run id", run.id)
print("\nOpen Datasets → Compare in the UI for the side-by-side view (quality + cost + latency).")

## What this loop showed

| variant | lever | what happened |
|---|---|---|
| V1 baseline | terse prompt, bare schema | floor; `object_id` comes out as `"1"`, `"2"` |
| V2 schema-spec | field descriptions | ID formats improve; the win is significant and cheap |
| V3 prompt-rules | explicit rules | better object judgment; `policyholder_name` regresses |
| V3_fixed | new prompt version | names recover; prove it on `policy_acc` |
| V4 few-shot (×2) | one example | apparent win does not replicate; extra cost does |
| V5 decomposition | four calls | worse quality, ~2× cost |
| V6 gpt-4o | larger model | better on hard objects; most gains within noise at ~20× cost |

Define correctness for a nested object before you optimize. Score in layers: parse gate, typed fields, object F1, presence, cost, latency. At small n, prove the delta with a paired test and a replication. Use traces to decide the next change.

After you pick a variant, grow the dataset from production traces. n=30 cannot separate close calls. Put `run_round` in CI so prompt edits cannot regress silently.
